# 👾Qwen2大模型微调入门

作者：林泽毅

教程文章：https://zhuanlan.zhihu.com/p/702491999  

显存要求：10GB左右  

实验过程看：https://swanlab.cn/@ZeyiLin/Qwen2-fintune/runs/cfg5f8dzkp6vouxzaxlx6/chart

## 1.安装环境

本案例测试于modelscope==1.14.0、transformers==4.41.2、datasets==2.18.0、peft==0.11.1、accelerate==0.30.1、swanlab==0.3.9

In [ ]:
%pip install torch swanlab modelscope transformers datasets peft pandas accelerate

如果是第一次使用SwanLab，则前往[SwanLab](https://swanlab.cn)注册账号后，在[用户设置](https://swanlab.cn/settings/overview)复制API Key，如果执行下面的代码：

In [ ]:
!swanlab login

## 2. 数据集加载

1. 在[zh_cls_fudan-news - modelscope](https://modelscope.cn/datasets/huangjintao/zh_cls_fudan-news/files)下载train.jsonl和test.jsonl到同级目录下。
数据集图片

In [2]:
from modelscope import MsDataset
# dataset = MsDataset.load('swift/zh_cls_fudan-news', split='train')
# test_dataset = MsDataset.load('swift/zh_cls_fudan-news', subset_name='test', split='test')
# print(dataset)
# print(test_dataset)
"""
Dataset({
    features: ['text', 'category', 'output'],
    num_rows: 4000
})
Dataset({
    features: ['text', 'category', 'output'],
    num_rows: 959
})
"""

/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


"\nDataset({\n    features: ['text', 'category', 'output'],\n    num_rows: 4000\n})\nDataset({\n    features: ['text', 'category', 'output'],\n    num_rows: 959\n})\n"

2. 将train.jsonl和test.jsonl进行处理，转换成new_train.jsonl和new_test.jsonl

In [1]:
# 2.将train.jsonl和test.jsonl进行处理，转换成new_train.jsonl和new_test.jsonl

import json
import pandas as pd
import os

def dataset_jsonl_transfer(origin_path, new_path):
    """
    将原始数据集转换为大模型微调所需数据格式的新数据集
    """
    messages = []

    # 读取旧的JSONL文件
    with open(origin_path, "r") as file:
        for line in file:
            # 解析每一行的json数据
            data = json.loads(line)
            context = data["text"]
            catagory = data["category"]
            label = data["output"]
            message = {
                "instruction": "你是一个文本分类领域的专家，你会接收到一段文本和几个潜在的分类选项，请输出文本内容的正确类型",
                "input": f"文本:{context},类型选型:{catagory}",
                "output": label,
            }
            messages.append(message)

    # 保存重构后的JSONL文件
    with open(new_path, "w", encoding="utf-8") as file:
        for message in messages:
            file.write(json.dumps(message, ensure_ascii=False) + "\n")


# 加载、处理数据集和测试集
train_dataset_path = "../datasets/train.jsonl"
test_dataset_path = "../datasets/test.jsonl"

train_jsonl_new_path = "../datasets/new_train.jsonl"
test_jsonl_new_path = "../datasets/new_test.jsonl"

if not os.path.exists(train_jsonl_new_path):
    dataset_jsonl_transfer(train_dataset_path, train_jsonl_new_path)
if not os.path.exists(test_jsonl_new_path):
    dataset_jsonl_transfer(test_dataset_path, test_jsonl_new_path)

train_df = pd.read_json(train_jsonl_new_path)  # 取前1000条做训练（可选）
test_df = pd.read_json(test_jsonl_new_path)[:10]  # 取前10条做主观评测

In [2]:
import json
import pandas as pd
import os
train_jsonl_new_path = "/home/ubuntu/why/jbdataprocess/negative_train.json"
test_jsonl_new_path = "/home/ubuntu/why/jbdataprocess/negative_dev.json"
train_df = pd.read_json(train_jsonl_new_path)  # 取前1000条做训练（可选）
test_df = pd.read_json(test_jsonl_new_path)[:10]  # 取前10条做主观评测

## 3. 下载/加载模型和tokenizer

In [3]:
from modelscope import snapshot_download, AutoTokenizer
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
import torch

# 在modelscope上下载Qwen模型到本地目录下
# model_dir = snapshot_download("./qwen/Qwen2-1.5B-Instruct", cache_dir="../bmodel/", revision="master")

# Transformers加载模型权重
# tokenizer = AutoTokenizer.from_pretrained("../bmodel/qwen/Qwen2-1___5B-Instruct/", use_fast=False, trust_remote_code=True)
# model = AutoModelForCausalLM.from_pretrained("../bmodel/qwen/Qwen2-1___5B-Instruct/", device_map="auto", torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("/home/ubuntu/bbmodel/Qwen/Qwen2___5-3B-Instruct", use_fast=False, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("/home/ubuntu/bbmodel/Qwen/Qwen2___5-3B-Instruct", device_map="auto", torch_dtype=torch.bfloat16)
model.enable_input_require_grads()  # 开启梯度检查点时，要执行该方法

/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.07s/it]


## 4. 预处理训练数据

In [4]:
def process_func(example):
    """
    将数据集进行预处理
    """
    MAX_LENGTH = 1024
    for message in example['messages']:
        if message['role'] == 'user':
            users=message['content']
        elif message['role'] == 'assistant':
            assistants=message['content']
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer(
        f"<|im_start|>system\n你是物业管理者的助手，负责筛查群聊对话中，负面情绪强烈的物业异常事件，进行向上汇报。<|im_end|>\n<|im_start|>user\n{users}<|im_end|>\n<|im_start|>assistant\n",
        add_special_tokens=False,
    )
    response = tokenizer(f"{assistants}", add_special_tokens=False)
    input_ids = (
        instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    )
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]
    labels = (
        [-100] * len(instruction["input_ids"])
        + response["input_ids"]
        + [tokenizer.pad_token_id]
    )
    if len(input_ids) > MAX_LENGTH:  # 做一个截断
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
train_dataset = train_ds.map(process_func, remove_columns=train_ds.column_names)

Map: 100%|██████████| 482/482 [00:03<00:00, 158.92 examples/s]


## 5. 设置LORA

In [5]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    inference_mode=False,  # 训练模式
    r=8,  # Lora 秩
    lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1,  # Dropout 比例
)

model = get_peft_model(model, config)

## 6. 训练

In [6]:
args = TrainingArguments(
    output_dir="./output/Qwen2",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    logging_steps=10,
    num_train_epochs=2,
    save_steps=100,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none",
)

In [7]:
from swanlab.integration.huggingface import SwanLabCallback
import swanlab

swanlab_callback = SwanLabCallback(
    project="Qwen2-fintune",
    experiment_name="Qwen2-1.5B-Instruct",
    description="使用通义千问Qwen2-1.5B-Instruct模型在zh_cls_fudan-news数据集上微调。",
    config={
        "model": "/home/ubuntu/bbmodel/Qwen/Qwen2___5-3B-Instruct",
        "dataset": "/home/ubuntu/why/jbdataprocess/negative_train.json",
    },
)

In [8]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    callbacks=[swanlab_callback],
)

trainer.train()


swanlab.log({"Prediction": test_text_list})
swanlab.finish()

    



swanlab: swanlab version 0.3.21 is available!  Upgrade: `pip install -U swanlab`
swanlab: Tracking run with swanlab version 0.3.16                                  
swanlab: Run data will be saved locally in /home/ubuntu/why/swanlog/run-20240929_163846-a3b1799d
swanlab: 👋 Hi whytehighmore, welcome to swanlab!
swanlab: Syncing run Qwen2-1.5B-Instruct_Sep29_16-38-46 to the cloud
swanlab: 🌟 Run `swanlab watch -l /home/ubuntu/why/swanlog` to view SwanLab Experiment Dashboard locally
swanlab: 🏠 View project at https://swanlab.cn/@whytehighmore/Qwen2-fintune
swanlab: 🚀 View run at https://swanlab.cn/@whytehighmore/Qwen2-fintune/runs/37sjtb22wflm4jbrhrx6h


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
10,123.203100
20,3.416600
30,17.818900
40,0.498600
50,0.361200
60,0.537400


swanlab: Step 60 on key train/epoch already exists, ignored.


In [11]:
# ====== 训练结束后的预测 ===== #

def predict(messages, model, tokenizer):
    device = "cuda"
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    generated_ids = model.generate(model_inputs.input_ids, max_new_tokens=512)
    generated_ids = [
        output_ids[len(input_ids) :]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(response)


    return response

test_text_list = []
test_df_less = test_df[0:2] 
for index, row in test_df.iterrows():
    # input_value = row["instruction"]
    # input_value = row["input"]
    for rows in row['messages']:
        if rows['role'] == 'user':
            input_value=rows['content']
        elif rows['role'] == 'system':
            instruction=rows['content']
    messages = [
        {"role": "system", "content": f"{instruction}"},
        {"role": "user", "content": f"{input_value}"},
    ]

    response = predict(messages, model, tokenizer)
    messages.append({"role": "assistant", "content": f"{response}"})
    result_text = f"{messages[0]}\n\n{messages[1]}\n\n{messages[2]}"
    test_text_list.append(swanlab.Text(result_text, caption=response))

print('over')

/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


[
    {
        "话题倾向": ""
    }
]


/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


[
    {
        "话题倾向": ""
    }
]


/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


...

/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


牵手(客户)不是一个人，无法判定第一。


/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/ubuntu/miniconda3/envs/tenv2/lib/python3.10/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


[
    {
        "话题倾向": ""
    }
]
[
    {
        "话题倾向": ""
    }
]
[
    {
        "话题倾向": ""
    }
]
[
    {
        "话题倾向": ""
    }
]
[
    {
        "话题倾向": ""
    }
]
[
    {
        "话题倾向": ""
    }
]
[
    {
        "话题倾向": ""
    }
]
[
    {
        "话题倾向": ""
    }
]
[
    {
        "话题倾向": ""
    }
]
over


: 

## 量化

In [9]:
for index, row in test_df.iterrows():
    # input_value = row["instruction"]
    # input_value = row["input"]
    for rows in row['messages']:
        if rows['role'] == 'user':
            input_value=rows['content']
        elif rows['role'] == 'system':
            instruction=rows['content']
    messages = [
        {"role": "system", "content": f"{instruction}"},
        {"role": "user", "content": f"{input_value}"},
    ]

print()





KeyError: 'role'